# Full syllabus training pipeline

Runs end-to-end from the freshman index CSV through LoRA fine-tuning:

1. **csv** — `us_freshman_core_syllabi_index.csv` → URL JSONL  
2. **ingest** — fetch each `source_url`, extract PDF/HTML text  
3. **process** — regex + heuristic entities JSONL  
4. **build** — chat-format train/valid JSONL for SFT  
5. **train** — `trl.SFTTrainer` + MLflow (optional)

Uses [`pipeline_runner.py`](../pipeline_runner.py) so paths and steps stay consistent with the CLI scripts.

## Prerequisites

- Open this notebook with the **project root** (`training_pipeline`) on `sys.path` and as the working directory (the first code cell does this).
- For **training**, use a GPU runtime (local Jupyter or Colab) and install the `train` extra: `pip install -e ".[train]"` from the repo root, or the Colab cell below.
- **Ingest** performs live HTTP requests; use `INGEST_MAX_ROWS` for a quick smoke test.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/training_pipeline").resolve()
    if not REPO_ROOT.is_dir():
        raise FileNotFoundError(
            "Expected the repo at /content/training_pipeline. Clone it there or adjust REPO_ROOT."
        )
else:
    from pipeline_runner import find_repo_root

    REPO_ROOT = find_repo_root()

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)
print("IN_COLAB:", IN_COLAB)

## Optional: Colab dependency install

Uncomment or run after cloning the repo into `/content/training_pipeline`.

In [ ]:
# if IN_COLAB:
#     %pip install -q accelerate datasets mlflow peft transformers trl pymupdf requests duckduckgo-search
#     # Optional GPU wheels for Colab (pick a CUDA index matching your runtime):
#     # %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print("Skip Colab installs unless you are on Colab and need them.")

## Configure and run

- Set `INGEST_MAX_ROWS` to a small integer (e.g. `3`) to smoke-test without fetching all URLs.
- Set `STEPS` to omit `train` while debugging earlier stages.
- Adjust `MLFLOW_TRACKING_URI` or set `cfg.disable_mlflow = True` if you do not want MLflow.

In [ ]:
from pipeline_runner import SyllabusPipelineConfig, run_syllabus_training_pipeline

cfg = SyllabusPipelineConfig.for_repo(REPO_ROOT)

INGEST_MAX_ROWS = 3  # set to None to fetch all rows from the URL JSONL
cfg.ingest_max_rows = INGEST_MAX_ROWS
cfg.ingest_sleep_seconds = 1.0

STEPS = ("csv", "ingest", "process", "build", "train")
# STEPS = ("csv", "ingest", "process", "build")  # skip training

run_syllabus_training_pipeline(cfg, steps=STEPS)

## Outputs (defaults)

| Step | Path |
|------|------|
| URL JSONL | `data/ingested/us_freshman_core_syllabi_urls.jsonl` |
| With text | `data/ingested/us_freshman_core_syllabi_with_text.jsonl` |
| Entities | `data/labeled/syllabus_entities.jsonl` |
| SFT train/valid | `data/finetune/train.jsonl`, `data/finetune/valid.jsonl` |
| Model | `artifacts/hf_syllabus_extractor/` |
| MLflow (default local) | `mlruns/` (URI from `MLFLOW_TRACKING_URI` or `file://…/mlruns`) |

The older Colab-focused notebook [`train_syllabus_extractor_colab.ipynb`](train_syllabus_extractor_colab.ipynb) still covers upload-your-own JSONL + train only.